## Setup

### Before you start

Make sure GPU accelerator is enabled: in Kaggle, go to Notebook Settings (right sidebar) -> Accelerator -> select GPU (P100 or T4x2), and turn Internet ON.

In [ ]:
!nvidia-smi

**NOTE:** To make it easier for us to manage datasets, images and models we create a `HOME` constant.

In [ ]:
import os
HOME = "/kaggle/working"
print(HOME)

## Install YOLO12 via Ultralytics

In [ ]:
%pip install -U ultralytics supervision roboflow -q
import ultralytics
ultralytics.checks()
print(ultralytics.__version__)

**IMPORTANT:** After running the install cell above for the first time (or after upgrading), **restart the kernel** (Kaggle: Run > Restart Session) before continuing. This ensures the old `ultralytics` module isn't still cached in memory, which can cause `AttributeError: Can't get attribute 'A2C2f'`. After restarting, run all cells from the top again (including the `HOME` cell).

## Fine-tune YOLO12 on dataset

In [ ]:
!mkdir -p {HOME}/datasets
%cd {HOME}/datasets

from roboflow import Roboflow
rf = Roboflow(api_key="-----------------")
project = rf.workspace("----------------").project("---------------------")
version = project.version(1)
dataset = version.download("------------")

## Custom Training

In [ ]:
%cd {HOME}

!yolo task=detect mode=train model=yolo12s.pt data={dataset.location}/data.yaml epochs=50 imgsz=640 plots=True

**NOTE:** The results of the completed training are saved in `{HOME}/runs/detect/train/`. Let's examine them.

In [ ]:
!ls {HOME}/runs/detect/train/

In [ ]:
from IPython.display import Image as IPyImage

IPyImage(filename=f'{HOME}/runs/detect/train/confusion_matrix.png', width=600)

In [ ]:
from IPython.display import Image as IPyImage

IPyImage(filename=f'{HOME}/runs/detect/train/results.png', width=600)

In [ ]:
from IPython.display import Image as IPyImage

IPyImage(filename=f'{HOME}/runs/detect/train/val_batch0_pred.jpg', width=600)

## Validate fine-tuned model

In [ ]:
!yolo task=detect mode=val model={HOME}/runs/detect/train/weights/best.pt data={dataset.location}/data.yaml

Validation Confusion Matrix

In [ ]:
from IPython.display import Image as IPyImage

IPyImage(filename=f'{HOME}/runs/detect/val/confusion_matrix.png', width=600)

In [ ]:
from IPython.display import Image as IPyImage

IPyImage(filename=f'{HOME}/runs/detect/val/val_batch0_pred.jpg', width=600)

In [ ]:
from IPython.display import Image as IPyImage

IPyImage(filename=f'{HOME}/runs/detect/val/val_batch1_pred.jpg', width=600)

## Inference with custom model

In [ ]:
!yolo task=detect mode=predict model={HOME}/runs/detect/train/weights/best.pt conf=0.25 source={dataset.location}/test/images save=True

**NOTE:** Let's take a look at few results.

In [ ]:
import glob
import os
from IPython.display import Image as IPyImage, display

latest_folder = max(glob.glob(f'{HOME}/runs/detect/predict*/'), key=os.path.getmtime)
for img in sorted(glob.glob(f'{latest_folder}/*.jpg'))[:5]:
    display(IPyImage(filename=img, width=600))
    print("\n")

## Archive results

In [ ]:
import shutil

shutil.make_archive(f'{HOME}/detect_results', "zip", f'{HOME}/runs/detect')